# Module 9: Deploy Your Agent on the Inference You Own (optional capstone)

This is the optional capstone. Run it if there is time. Omer handed you back a tuned inference layer. Across the optimization labs you served the model, tested quantization and speculative decoding, drove the engine to saturation, and chose an operating point. Now you put an agent on top of it. It deploys into your namespace as a plain Kubernetes Deployment, calls the [vLLM](https://docs.vllm.ai) you tuned, and its traffic lands in the same metrics you read all day. The model answering every turn is the server you own, on your GPU, not a rented API.

## Learning objectives
- See why an agent's wall-clock time is almost entirely inference, not tool or CPU work
- Deploy an agent as a plain Kubernetes Deployment and Service in your own namespace
- Point the agent at your in-namespace vLLM by its Service name, with no cross-namespace setup
- Reach the agent with kubectl port-forward and send it questions in and out of scope
- Watch the agent reason with thinking on and make a real tool call, all on the model you serve
- Fire several agents at once and watch them batch on the server and land in your vLLM metrics
- Know where this goes in production: scaling replicas and scaling nodes

## Prerequisites
- Finished the optimization labs (Modules 5 through 8), with a working vLLM Service named `vllm` in your namespace
- A live cluster with `kubectl` and your namespace-scoped kubeconfig
- About 15 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat) &middot; [Kubernetes Deployments](https://kubernetes.io/docs/concepts/workloads/controllers/deployment/) &middot; [kubectl port-forward](https://kubernetes.io/docs/reference/generated/kubectl/kubectl-commands#port-forward) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Agent on owned inference design basics

An agent is just a client of a model. It runs a loop: call the model, maybe run a tool, call the model again with the result. The only question this module answers is where that model lives. Here it lives in your namespace, on the GPU you tuned, and the agent is an ordinary container next to it.

- **A plain Deployment.** The agent is a small HTTP service (`agent/agent.py`): it wraps each request with the solutions-architect system prompt from Module 1, gives the model one tool that looks up Akamai GPU pricing, and runs with Qwen3 thinking on so it reasons before it acts. No framework, no controller, no custom resources. It runs with a stock `python:3.12-slim` image and installs the one dependency it needs at start.
- **Same namespace, short name.** The agent and your vLLM share a namespace, so the agent reaches the model at the bare Service name `http://vllm:8000/v1`. There is no cross-namespace FQDN and nothing cluster-scoped to install, so your scoped kubeconfig can deploy the whole thing.
- **The persona and the tool.** The system prompt makes it an Akamai Cloud solutions architect: tactical, in scope on Akamai compute, LKE, storage, networking, GPUs, and inference, and honest about what it does not cover. Its one tool, `akamai_gpu_pricing`, turns a cost question into a real tool call the model reasons over.

![A small agent Deployment in your namespace calls your in-namespace vLLM Service on your GPU, and its traffic lands in the same metrics](images/09_agents_on_k8s_architecture.png)

## 1. Setup

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

In [ ]:
import os, sys, json, time, threading, subprocess, atexit
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

if Path("../common").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import get_settings, print_settings, build_client
from common import metrics, foundation

settings = get_settings()
settings.model_name = foundation.served_model_name(settings)  # use the model the server actually serves
print_settings(settings)
NS = settings.namespace
AGENT_DIR = REPO_ROOT / "09_optional_agents_on_k8s" if (REPO_ROOT / "09_optional_agents_on_k8s" / "agent").exists() else REPO_ROOT

**What you should see:** your resolved settings and namespace. The capstone reuses the same `common/config.py` every module uses.

## 2. Where does the agent's time go?

Before deploying, see where an agent's time actually goes. An agent is a loop of model calls with a little tool work between them. Run a short turn (two model calls with a tool call simulated between them) and split the wall-clock: the tool runs on the CPU and the model calls are inference over HTTP. Watch how lopsided the split is.

In [ ]:
# Requires a live vLLM endpoint. One tool-using turn, with the tool's CPU time measured.
client = build_client(settings)
tool_time = 0.0

def add(a, b):
    global tool_time
    t = time.perf_counter()
    result = a + b
    tool_time += time.perf_counter() - t
    return result

start = time.perf_counter()
# Turn 1: a model call. The tool call is simulated by hand between the two turns,
# which is enough to time CPU tool work against inference.
msgs = [
    {"role": "system", "content": "Use the add tool for arithmetic. Be brief."},
    {"role": "user", "content": "What is 47 + 58? Use the tool."},
]
client.chat.completions.create(model=settings.model_name, messages=msgs, max_tokens=80, temperature=0.0)
# Run the tool (CPU work), then feed the result back for the final answer.
result = add(47, 58)
msgs += [
    {"role": "assistant", "content": "calling add(47, 58)"},
    {"role": "user", "content": f"The tool returned {result}. Give the final answer in one sentence."},
]
client.chat.completions.create(model=settings.model_name, messages=msgs, max_tokens=80, temperature=0.0)
wall = time.perf_counter() - start

infer = wall - tool_time
print(f"agent wall time:   {wall:.2f} s")
print(f"  tool / CPU time: {tool_time * 1000:.3f} ms")
print(f"  inference time:  {infer:.2f} s  (the two model turns over HTTP, including the network round-trip)")
print(f"\ninference is {100 * infer / wall:.1f}% of the agent's wall time")

**What you should see:** the tool work rounding toward zero against seconds of inference, so inference is essentially 100% of the agent's wall time. That is the headline for this loop: the tool here is a CPU add that rounds to zero, so almost all the wall time is inference, through time to first token (Module 3) and tokens per second. A real tool that calls over the network or runs work adds its own latency on top, which you cannot tune here. What you can tune, and what dominates a reasoning-heavy agent that makes call after call, is inference. The inference layer you tuned is the agent's latency budget, which is why this module deploys the agent onto it.

## 3. The agent code

Print the agent so you can read what you are about to deploy. It is short on purpose: a system prompt and one call to your vLLM.

In [ ]:
print("The agent you will deploy (agent/agent.py):")
print("-" * 60)
print(open(AGENT_DIR / "agent" / "agent.py").read())

**What you should see:** the agent source. It reads `VLLM_BASE_URL` and `MODEL_NAME` from its environment and calls your vLLM with the OpenAI client. It registers one tool, `akamai_gpu_pricing`, and runs with Qwen3 thinking on, so the model reasons, calls the tool when a question needs a GPU price, reads the result, and answers. The persona and that one tool are the whole agent; everything else is plumbing.

## 4. Deploy the agent

Two objects go into your namespace: a ConfigMap holding the agent code, and the Deployment plus Service that runs it (`manifests/agent.yaml`). The Deployment mounts the code, installs the OpenAI client at start, and serves on port 8080. All of it is namespaced, so your scoped kubeconfig is enough.

In [ ]:
# Requires a live cluster and your namespace-scoped kubeconfig.
AGENT_PY = str(AGENT_DIR / "agent" / "agent.py")
MANIFEST = str(AGENT_DIR / "manifests" / "agent.yaml")

# 1) Put the agent code in a ConfigMap the Deployment mounts at /app (idempotent).
cm = subprocess.run(
    ["kubectl", "create", "configmap", "agent-code",
     f"--from-file=agent.py={AGENT_PY}", "-n", NS,
     "--dry-run=client", "-o", "yaml"],
    check=True, capture_output=True, text=True).stdout
subprocess.run(["kubectl", "apply", "-f", "-", "-n", NS], input=cm, text=True, check=True)

# 2) Deploy the agent and its Service.
subprocess.run(["kubectl", "delete", "deployment", "sa-agent", "-n", NS, "--ignore-not-found=true"], check=True)
subprocess.run(["kubectl", "apply", "-f", MANIFEST, "-n", NS], check=True)

# 3) Point the agent at the model your server actually serves, resolved from the environment
# (never hardcoded). This overrides the manifest default so the agent does not 404 on a mismatch.
subprocess.run(["kubectl", "set", "env", "deploy/sa-agent", f"MODEL_NAME={settings.model_name}", "-n", NS], check=True)

# 4) Wait for it to be Ready.
subprocess.run(["kubectl", "rollout", "status", "deploy/sa-agent", "-n", NS, "--timeout=180s"], check=True)
subprocess.run(["kubectl", "get", "pods", "-l", "app=agent,component=sa-agent", "-n", NS])

**What you should see:** the ConfigMap and Deployment applied, the rollout reaching Ready, and one `sa-agent` pod `Running`. The first start takes a few seconds to install the OpenAI client. If the pod is not Ready, `kubectl logs deploy/sa-agent -n $NS` shows why; a connection error to vLLM means your `vllm` Service is not up.

## 5. Talk to it

The agent has no public address, so port-forward its Service to your notebook and POST questions. Ask one thing in scope and one thing out of scope, and watch it route honestly instead of guessing. Every answer is generated by your vLLM.

In [ ]:
# Requires the agent Running. Port-forward the agent Service, then POST questions.
import requests

pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/sa-agent", "8080:8080", "-n", NS],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
atexit.register(pf.terminate)
time.sleep(4)  # let the port-forward establish

def ask(message):
    # Thinking is on and the agent may make a tool call, so allow a generous timeout.
    r = requests.post("http://localhost:8080", json={"message": message}, timeout=120)
    data = r.json()
    if r.status_code >= 400 or "error" in data:
        raise RuntimeError(data.get("error") or data)
    return data["answer"]

print("=== in scope ===")
print(ask("What should I check before adding a GPU node pool to an LKE cluster?"))
print("\n=== out of scope ===")
print(ask("How do I write an EdgeWorker for the CDN?"))
print("\n=== what powers it ===")
print(ask("What model and endpoint are you running on?"))
print("\n=== uses the pricing tool ===")
print(ask("To serve a 4B model on one card, what does an RTX 4000 Ada cost per month on Akamai, "
          "and when is the RTX PRO 6000 Blackwell worth the jump? Check the pricing."))

**What you should see:** a tactical checklist about LKE and GPU node pools for the first question, a polite redirect to the edge compute team for the second, the agent naming your model and your vLLM endpoint for the third, and, for the fourth, a cost answer that quotes real per-month prices because the agent called its `akamai_gpu_pricing` tool and reasoned over the result. Every answer came from your vLLM, on your GPU. These take a little longer than a plain chat because thinking is on. If the POST refuses or hangs, the port-forward did not establish; re-run the previous cell, or check `kubectl get svc sa-agent -n $NS`.

## 6. Many agents at once: batching, in your metrics

This is the payoff of the whole workshop. One agent uses a fraction of the GPU. Fire six agent questions at once and two things happen together: the server folds their model turns into one running batch (the continuous batching from Module 7), so six finish in far less than six times one, and that traffic appears in the same metrics you have read since Module 3. Sample the metrics while the six run.

In [ ]:
# Requires the agent up and the port-forward from section 5 still active.
# Six DISTINCT questions, so the speedup is continuous batching (Module 7), not prefix caching (Module 3).
QUESTIONS = [
    "In two sentences, what is a NodeBalancer and when do I use one?",
    "In two sentences, what is an Object Storage lifecycle policy good for?",
    "In two sentences, when should I add a GPU node pool to LKE?",
    "In two sentences, what is a VPC on Akamai Cloud for?",
    "In two sentences, what does Cloud Firewall protect?",
    "In two sentences, what is LKE and when should I use it?",
]

def run_session(i):
    t = time.perf_counter()
    ask(QUESTIONS[i])
    return time.perf_counter() - t

# One session first, as a baseline, with its own distinct prompt.
t0 = time.perf_counter()
ask("In two sentences, what is a Linode and how is it billed?")
single = time.perf_counter() - t0

# Now six at once, sampling the vLLM metrics while they run.
peak = {"running": 0, "kv": 0.0}
stop = threading.Event()

def sample():
    while not stop.is_set():
        try:
            s = metrics.snapshot(settings.metrics_url)
            peak["running"] = max(peak["running"], int(s["vllm:num_requests_running"]))
            peak["kv"] = max(peak["kv"], s["vllm:gpu_cache_usage_perc"])
        except Exception:
            pass
        time.sleep(0.2)

t = threading.Thread(target=sample, daemon=True); t.start()
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as pool:
    times = list(pool.map(run_session, range(6)))
six_wall = time.perf_counter() - start
stop.set(); t.join(timeout=2)

print(f"one session alone        : {single:.2f} s")
print(f"six sessions concurrently: {six_wall:.2f} s total, slowest {max(times):.2f} s")
print(f"peak vLLM requests running: {peak['running']}  (they batched together)")
print(f"peak KV cache usage       : {peak['kv'] * 100:.1f}%")
print("That load came from your agent, served by the vLLM you tuned.")

**What you should see:** six sessions finishing in far less than six times one, because the server batched them. The slowest of the six is a small multiple of a single call, not six times it, since decode is bandwidth-bound and the batch shares each weight read. You also see a peak of several concurrent requests running and a bump in KV cache usage. The six prompts differ, so what you see is continuous batching (Module 7), not the prefix caching from Module 3. The KV cache you sized in Module 2 is holding it, and the metrics you learned to operate are reporting on a real agent.

## 7. In production: scaling and scale-to-zero

You now run two things you control: a vLLM serving the model and an agent calling it. In production you would not run them flat out around the clock. Two separate loops handle that.

- **Scaling replicas.** Tools like KServe or Knative scale your inference and agent pods up under load and back down when traffic falls, including all the way to zero when nothing is calling. That trades a cold start for paying nothing while idle, which is the right deal for bursty traffic.
- **Scaling nodes.** A node autoscaler such as Karpenter or the cluster autoscaler adds and removes the GPU machines underneath, so when your pods scale to zero the expensive card can go away too.

This workshop deliberately did the opposite: one dedicated card, driven to saturation, tuned by hand, so you could see the limits. That is the right mode for predictable latency on steady traffic. Scale-to-zero is the other half of the story, for when the traffic is not steady. Same metrics, same vLLM, a different operating posture.

## Things to know

- **The agent is just a client.** It holds a system prompt and one tool, and calls your vLLM with the OpenAI client. The lesson is where the model runs, not the agent code. Swap in any framework you like; the model stays yours.
- **This is the Solutions Architect agent, kept minimal.** The same persona ships as the full [Akamai Solutions Architect Agent](https://github.com/akamai-developers/akamai-workshop-solution-architect-agent), which adds memory and an MCP that reaches your Akamai account. This capstone leaves the MCP out on purpose, so the deployed agent needs no credentials and cannot touch your account; its one tool reads a static pricing table. Point the full agent's `VLLM_BASE_URL` at the vLLM you tuned and it runs on inference you own.
- **Same namespace, short name.** Because the agent and vLLM share your namespace, the agent uses the short Service name `http://vllm:8000/v1`, with no cross-namespace FQDN and nothing cluster-scoped to install.
- **The agent is a new client through the firewall.** On a cluster that enforces a default-deny NetworkPolicy, your namespace needs a rule allowing `app: agent` to reach vLLM on 8000. The rule that lets your workspace reach vLLM does not cover the agent, because the agent is a different pod.
- **Nothing here needs cluster admin.** A Deployment, a Service, and a ConfigMap all live in your namespace, which your scoped kubeconfig manages.
- **Match the agent's timeouts to the server you tuned.** An agent calling your vLLM should fail in tiers. A total `request_timeout` (say 300 s) bounds the whole call. A first-token timeout (say 60 s) catches a stuck prefill on an overloaded server, the time to first token you measured in Module 3. An inter-token timeout (say 30 s) catches a generation that stalls mid-stream under a preemption cascade, the preemption from Module 7. The `ask()` helper here passes a single `timeout=120`, the blunt version of all three.

> NOTE: vLLM enforces no API key by default, so the agent sends `not-needed`. If you put auth in front of your vLLM, set `VLLM_API_KEY` on the Deployment and the same agent keeps working.

## Try it yourself

**Tighten the scope.** Edit the system prompt in `agent/agent.py`, re-create the ConfigMap, and `kubectl rollout restart deploy/sa-agent`. Re-ask an out-of-scope question and watch the routing change.

**Add another tool.** The agent already prices GPUs. Add a second function to `agent.py`, for example one that lists your pods, register it in `TOOLS`, and let the model choose between them. Now it reads your cluster instead of only describing it, still on the model you own.

## Summary

- An agent is a client of a model, and its wall-clock time is almost entirely inference. The whole point of this module is that its model is the vLLM you own, on your GPU.
- You deployed the agent as a plain Deployment and Service in your own namespace, with no CRDs, no controller, and no cluster admin.
- The agent reaches your vLLM by its short Service name, and you reached the agent with kubectl port-forward.
- You watched it reason with thinking on and make a real tool call to price an Akamai GPU, all answered by the model you serve.
- You drove six agents at once, watched them batch into close to the time of one, and saw their traffic land in the same vLLM metrics you tuned all workshop.
- In production you would scale replicas with KServe or Knative and nodes with an autoscaler; here you tuned one card on purpose, to see the limits.

## Done

You started by renting inference and ended with an agent answering on a GPU you tuned yourself. You own the whole stack now: the model, the server, the metrics, the tuning, and the agent on top of it.